# 05. Runtime Context

> 에이전트에는 **요청별로 바뀌는 정보**(사용자 ID, 세션 ID, 권한)가 흘러야 해요. Runtime이 Context · Store · Stream Writer를 어떻게 중개하는지 정리하고, 이 노트북에서는 특히 Context/Store/미들웨어 접근에 집중해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Runtime의 세 가지 구성 요소(Context, Store, Stream Writer)의 역할과 차이를 설명할 수 있어요
2. `dataclass`로 Context 스키마를 정의하고 에이전트 호출 시 주입하는 방법을 알아요
3. `ToolRuntime[ContextType]`을 사용해 도구 안에서 Context와 Store에 접근할 수 있어요
4. Stream Writer가 Runtime의 실시간 출력 통로임을 이해하고, 자세한 스트리밍 실습은 이전 노트북과 연결할 수 있어요
5. 미들웨어에서 Runtime을 활용해 동적 프롬프트, 권한 검사, 사용량 추적을 구현할 수 있어요

## 사전 지식

- Part 05의 `01-Create-Agent.ipynb` — `create_agent`, `AgentState`
- Part 05의 `02-Tools-V1.ipynb` — `@tool`, `ToolRuntime`
- Part 05의 `04-Streaming-V1.ipynb` — `stream_mode` 종류

## Runtime이란?

LangChain V1의 `create_agent`는 **Runtime**이라는 실행 환경을 에이전트 내부에 제공해요.  
도구는 실행 도중 Runtime을 통해 Context, Store, Stream Writer에 접근할 수 있고, 미들웨어도 Runtime을 통해 요청별 실행 정보를 활용할 수 있어요.

> 🔑 **핵심 개념**: Runtime을 **호텔 프론트 데스크**에 비유할 수 있어요. 호텔 투숙객(도구/미들웨어)은 프론트 데스크(Runtime)를 통해 객실 정보(Context), 금고(Store) 같은 실행 자원에 접근해요. 도구는 필요할 때 룸서비스 전화(Stream Writer)처럼 진행 상황도 밖으로 알릴 수 있어요. 각 투숙객이 직접 호텔 시스템에 들어갈 필요 없이, 프론트 데스크가 서비스를 중개해줘요.

| 구성 요소 | 유형 | 역할 | 접근 방법 |
|-----------|------|------|-----------|
| **Context** | 정적 | 사용자 ID, 언어, 사용자 등급 등 요청별 메타데이터 | `runtime.context` |
| **Store** | 동적 | 세션을 넘는 장기 메모리 읽기/쓰기 | `runtime.store.get()` / `.put()` |
| **Stream Writer** | 실시간 | 도구 진행 상황을 커스텀 스트림으로 전송 | `runtime.stream_writer(dict)` |

### Context vs State vs Store 비교

| 특성 | Context | State | Store |
|------|---------|-------|-------|
| **변경 가능** | 불변 (읽기 전용) | 변경 가능 (add_messages 등) | 변경 가능 (put/get) |
| **지속 범위** | 단일 invoke 호출 | 대화 세션 (thread_id) | 영구 (세션 간 공유) |
| **주입 방법** | `agent.invoke(..., context=)` | `agent.invoke({"messages": ...})` | `create_agent(store=)` |
| **대표 데이터** | user_id, 권한, API 키 | 메시지, 인증 상태 | 사용자 선호도, 검색 기록 |

> 💡 **실무 팁**: 도구 파라미터에 `ToolRuntime`을 넣으면 LLM에는 보이지 않고 에이전트 실행 엔진이 자동으로 값을 채워줘요.

```mermaid
flowchart LR
    %% 왼쪽에서 오른쪽으로 흐름을 나누어 선 겹침을 줄입니다.
    subgraph Request["요청"]
        direction TB
        User([사용자 요청])
        InputCtx["context=...<br/>(요청 메타데이터)"]
    end

    subgraph Execution["에이전트 내부 실행"]
        direction TB
        Agent[에이전트 실행]
        Tool[도구 실행]
        MW[미들웨어]
        Agent --> Tool
        Agent --> MW
    end

    subgraph RuntimeBox["Runtime<br/>(호텔 프론트 데스크)"]
        direction TB
        Runtime["ToolRuntime<br/>request.runtime"]
    end

    subgraph Resources["Runtime이 제공하는 자원"]
        direction TB
        Ctx["Context<br/>(정적 메타데이터)"]
        Store[("Store<br/>(장기 메모리)")]
        SW["Stream Writer<br/>(실시간 업데이트)"]
    end

    User --> Agent
    InputCtx -->|invoke 시 주입| Agent
    Tool -->|ToolRuntime| Runtime
    MW -->|request.runtime| Runtime
    Runtime -->|context| Ctx
    Runtime -->|store| Store
    Runtime -->|stream_writer| SW

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef runtime fill:#f8f9fa,stroke:#6c757d,color:#212529
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class User,InputCtx input
    class Agent,Tool,MW process
    class Runtime runtime
    class Ctx,Store storage
    class SW output
```

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 OPENAI_API_KEY 등을 읽어와요)
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (선택 사항: 실행 흐름을 LangSmith 대시보드에서 확인할 수 있어요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part05-Runtime"

In [ ]:
# 기본 모델 초기화 — 모든 예제에서 공통으로 사용해요
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# Anthropic 사용 시: "anthropic:claude-sonnet-4-5"
# Ollama 로컬 사용 시: "ollama:llama3"
model = init_chat_model("openai:gpt-4o-mini")

---

## 1. Context: 요청별 정적 메타데이터 주입

Context는 에이전트가 실행되는 동안 변경되지 않는 **요청별 메타데이터**예요.  
사용자 ID, 언어 설정, 권한 정보 등 "이 요청을 누가 어떤 상황에서 보냈는가"를 담아요.

### Context 정의와 주입 흐름

```python
@dataclass
class UserContext:
    user_id: str
    user_name: str
    language: str
```

1. `@dataclass`로 Context 스키마를 정의해요  
2. `create_agent(..., context_schema=UserContext)`로 에이전트에 등록해요  
3. `agent.invoke(..., context=UserContext(...))` 호출 시 값을 주입해요

여기까지는 **Context를 준비하고 호출에 실어 보내는 단계**예요.  
그 다음 질문은 "도구가 실행될 때 이 값을 어떻게 읽을까?"예요.

### 도구 안에서 Context를 읽는 방법

도구 함수는 LLM이 직접 실행하는 함수가 아니라, **LangChain 실행 엔진이 호출하는 함수**예요.  
그래서 도구 안에서 Context를 읽으려면 실행 엔진이 넣어주는 `runtime` 객체를 받아야 해요. 이때 사용하는 타입이 `ToolRuntime[UserContext]`예요.

```python
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    user_name = runtime.context.user_name
    return f"현재 사용자는 {user_name}입니다."
```

즉, 흐름은 이렇게 나뉘어요.

| 단계 | 코드 | 의미 |
|------|------|------|
| Context 모양 정의 | `class UserContext: ...` | 어떤 메타데이터를 받을지 정해요 |
| Agent에 등록 | `context_schema=UserContext` | 이 에이전트가 받을 Context 타입을 알려줘요 |
| 호출 시 주입 | `context=UserContext(...)` | 이번 요청의 실제 사용자 정보를 넘겨요 |
| 도구에서 읽기 | `runtime.context.user_name` | 실행 중인 도구가 Context를 꺼내 써요 |

> 💡 **핵심 정리**: `ToolRuntime`은 "도구가 Runtime에 접근하기 위한 통로"예요.  
> LLM에게 보이는 도구 입력이 아니라 LangChain이 자동으로 주입하는 실행 객체라서, 도구 schema에서는 제외돼요.

> 💡 **파라미터 구분**: `query: str`, `city: str`처럼 LLM이 채울 값은 일반 파라미터로 두고,  
> `runtime: ToolRuntime[...]`처럼 실행 엔진이 채울 값은 별도로 둬요. 예제에서는 읽기 쉽게 보통 마지막에 둡니다.


In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 생성: context_schema를 지정해야 Context 주입이 활성화돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → model → {tools_condition} → tools 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 코드는 그대로 — Context만 바꿔서 다른 사용자 시뮬레이션
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 2. Store: 세션을 넘는 장기 메모리

Context가 요청 안에서만 사는 정적 데이터라면, **Store**는 여러 대화 세션에 걸쳐 데이터를 **읽고 쓸 수 있는** 영구 저장소예요.

### Store의 데이터 모델

| 개념 | 설명 | 예시 |
|------|------|------|
| **Namespace** | 튜플 형태의 논리 파티션 | `("users",)`, `("history", "user_123")` |
| **Key** | 네임스페이스 안의 항목 식별자 | `"user_123"`, `"prefs"` |
| **Value** | 저장할 dict 데이터 | `{"preferences": "간결한 스타일"}` |

```python
# 저장
runtime.store.put(("users",), "user_123", {"preferences": "간결한 스타일"})

# 조회 — 없으면 None 반환
item = runtime.store.get(("users",), "user_123")
value = item.value  # {"preferences": "간결한 스타일"}
```

> 💡 **실무 팁**: 개발/테스트 환경에서는 `InMemoryStore`를, 프로덕션에서는 `PostgresSaver` 등 영속 백엔드로 교체해요.  
> `create_agent(..., store=store)` 인자만 바꾸면 되므로 코드 수정이 최소화돼요.

> ⚠️ **자주 하는 실수**: `runtime.store.get()` 결과가 `None`일 수 있어요.  
> `if item := runtime.store.get(...):` 패턴으로 None 체크를 항상 합니다.

In [ ]:
from langgraph.store.memory import InMemoryStore

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 3. Stream Writer: Runtime 안의 실시간 출력 통로

Stream Writer는 Runtime이 제공하는 세 번째 자원이에요.  
다만 `stream_mode="custom"`, 진행률 바, 다중 스트리밍 모드는 이미 `04-Streaming-V1.ipynb`에서 자세히 다뤘으므로, 여기서는 **Runtime 관점에서 어디에 붙어 있는지**만 짚고 넘어가요.

### Runtime 관점에서 보는 흐름

1. 도구가 `ToolRuntime`을 통해 현재 실행의 Runtime 객체를 받아요  
2. 도구 내부에서 `runtime.stream_writer({...})`로 커스텀 이벤트를 내보내요  
3. 호출자는 `.stream(..., stream_mode="custom")`으로 그 이벤트를 받아요

```python
@tool
def long_running_tool(runtime: ToolRuntime) -> str:
    runtime.stream_writer({"stage": "working"})
    return "done"
```

> 💡 **핵심 정리**: `stream_writer`는 새로운 저장소나 상태가 아니라, **도구 실행 중간 이벤트를 밖으로 내보내는 Runtime의 출력 채널**이에요.  
> 실제 진행률 표시, `updates + custom` 다중 모드, UI 출력 패턴은 이전 `04-Streaming-V1.ipynb`의 예제를 재사용하면 돼요.


---

## 4. 미들웨어에서 Runtime 접근

미들웨어도 도구와 동일하게 Runtime에 접근할 수 있어요.  
미들웨어에서 Context를 읽으면 **사용자별로 프롬프트나 동작을 다르게 만들 수 있어요**.

### 미들웨어 종류별 접근 방식

| 미들웨어 데코레이터 | Runtime 접근 방법 | 주요 용도 |
|--------------------|-------------------|-----------|
| `@dynamic_prompt` | `request.runtime.context` | 사용자별 시스템 프롬프트 생성 |
| `@before_model` | `runtime: Runtime[ContextType]` 파라미터 | 모델 호출 전 로깅/검증 |
| `@after_model` | `runtime: Runtime[ContextType]` 파라미터 | 응답 후 비용 추적/변환 |
| `@before_agent` | `runtime: Runtime[ContextType]` 파라미터 | 권한 검사, 조기 종료 |

> 🔑 **핵심 개념**: 미들웨어 파이프라인 실행 순서는  
> `@before_agent` → `@dynamic_prompt` → `@before_model` → 모델 호출 → `@after_model`  
> 이 순서를 이해하면 어떤 미들웨어에서 어떤 처리를 해야 할지 명확해져요.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. @before_agent: 권한 검사와 조기 종료

`@before_agent` 미들웨어는 에이전트 루프 진입 전에 실행돼요.  
권한이 없는 요청을 **에이전트가 LLM을 호출하기도 전에 차단**할 수 있어서 비용을 절약해요.

`can_jump_to=["end"]` 옵션을 지정하면 미들웨어가 `"jump_to": "end"`를 반환해서 즉시 종료할 수 있어요.

> 💡 **핵심 정리**: `@before_agent`에서 `{"jump_to": "end", "messages": [...]}` 를 반환하면  
> LLM 호출 없이 바로 응답을 반환해요. API 비용을 0으로 만드는 가장 효율적인 차단 방식이에요.

> 💡 **실무 팁**: 인증 토큰 검증, 요금제 사용량 초과 차단, 욕설 필터 등에 `@before_agent`를 활용해요.  
> 비즈니스 규칙은 LLM에게 맡기지 말고 코드 레벨에서 명확히 처리하는 게 안전해요.

In [ ]:
from langchain.agents.middleware import before_agent
from typing import Any

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 6. 종합 예제: 사용자 등급 기반 에이전트

지금까지 배운 Context, Store, 미들웨어를 조합한 실전 예제예요.  
사용자 등급(`free` / `premium` / `enterprise`)에 따라 도구 응답과 프롬프트가 달라지고,  
검색 기록은 Store에 저장되며, 사용량은 미들웨어가 추적해요.

> 💡 **핵심 정리**: 이 패턴은 SaaS 제품에서 매우 자주 사용해요.  
> 비즈니스 로직(등급별 기능 제한)을 LLM 프롬프트 안에 넣지 않고,  
> Context + 미들웨어로 코드 레벨에서 명확하게 처리하는 게 핵심이에요.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model
from langgraph.store.memory import InMemoryStore

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## TODO: 실습 — 나만의 Context 확장하기

아래 코드를 수정해서 새로운 Context 필드를 추가하고, 도구 동작을 바꿔보세요.

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: region Context 확장하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Runtime 3요소**: Context(정적 메타), Store(영속 메모리), Stream Writer(실시간 출력 채널)가 Runtime을 통해 도구/미들웨어와 연결돼요
- **Context 주입 패턴**: `@dataclass`로 스키마 정의 → `context_schema=` 등록 → 호출 시 `context=` 주입 → 도구에서 `ToolRuntime[T]`로 접근
- **ToolRuntime 자동 주입**: `ToolRuntime` 파라미터는 LLM에 노출되지 않고 에이전트 엔진이 자동으로 값을 채워줘요
- **Store CRUD**: `put(namespace, key, value)`로 쓰고 `get(namespace, key)`로 읽어요. `None` 체크 필수
- **Stream Writer 위치**: 이 노트북에서는 Runtime의 출력 채널이라는 역할만 짚고, 자세한 `stream_mode="custom"` 실습은 `04-Streaming-V1.ipynb`로 분리했어요
- **미들웨어 활용**: `@dynamic_prompt`는 요청별 시스템 프롬프트, `@before_model`/`@after_model`은 모델 호출 전후 처리, `@before_agent`는 권한 검사와 조기 종료에 사용해요
- **비즈니스 로직 분리**: 등급별 기능 제한, 사용량 추적 등 비즈니스 규칙은 LLM 프롬프트가 아닌 Context + 미들웨어 코드 레벨에서 처리해요

---


## 다음 노트북 예고

다음 `06-MCP-Server-Basics.ipynb`에서는 **MCP(Model Context Protocol) 서버 만들기**를 배워요. `FastMCP`와 `@mcp.tool()` 데코레이터로 도구를 서버로 노출하고, stdio / HTTP 전송 방식을 선택하는 방법, MCP Inspector로 서버를 가시화하는 방법까지 다뤄요. 에이전트에 붙일 도구를 독립 서비스로 분리해 여러 팀과 공유할 수 있어요.